##### Install requirements

In [9]:
import pandas as pd
import requests 


##### Loading the CSV file

In [10]:
takeoffs2016 = pd.read_csv("data/2016.csv")


In [11]:
takeoffs2017 = pd.read_csv("data/2017.csv")

In [12]:
takeoffs2018 = pd.read_csv("data/2018.csv")

#### Create Launch Sites

In [13]:
print(takeoffs2016.head())

      FL_DATE OP_CARRIER  OP_CARRIER_FL_NUM ORIGIN DEST  CRS_DEP_TIME  \
0  2016-01-01         DL               1248    DTW  LAX          1935   
1  2016-01-01         DL               1251    ATL  GRR          2125   
2  2016-01-01         DL               1254    LAX  ATL          2255   
3  2016-01-01         DL               1255    SLC  ATL          1656   
4  2016-01-01         DL               1256    BZN  MSP           900   

   DEP_TIME  DEP_DELAY  TAXI_OUT  WHEELS_OFF  ...  CRS_ELAPSED_TIME  \
0    1935.0        0.0      23.0      1958.0  ...             309.0   
1    2130.0        5.0      13.0      2143.0  ...             116.0   
2    2256.0        1.0      19.0      2315.0  ...             245.0   
3    1700.0        4.0      12.0      1712.0  ...             213.0   
4    1012.0       72.0      63.0      1115.0  ...             136.0   

   ACTUAL_ELAPSED_TIME  AIR_TIME  DISTANCE  CARRIER_DELAY  WEATHER_DELAY  \
0                285.0     249.0    1979.0            NaN 

In [14]:
takeoffs2016['ORIGIN'].unique() 

<StringArray>
['DTW', 'ATL', 'LAX', 'SLC', 'BZN', 'BNA', 'JAX', 'MSP', 'MDT', 'SAV',
 ...
 'GCK', 'GST', 'AKN', 'DLG', 'ABI', 'GRI', 'EFD', 'SPN', 'PGD', 'ENV']
Length: 313, dtype: str

In [15]:
AIRPORTS = {
    'Atlanta ATL': (36.63,  84.43),
    'Los Angeles LAX': (28.50,  -80.57),
    'Pittsburgh PIT': (34.74, -120.57),
    'Seattle SEA': (45.92,   63.34),
    'Dallas Fort Worth DFW': (5.24,  -52.77),
    'Denver DEN': (13.73,   80.23),
}

print(f'Airports being used: {len(AIRPORTS)}')
for name, (lat, lon) in AIRPORTS.items():
    print(f'  {name}: ({lat}, {lon})')

Airports being used: 6
  Atlanta ATL: (36.63, 84.43)
  Los Angeles LAX: (28.5, -80.57)
  Pittsburgh PIT: (34.74, -120.57)
  Seattle SEA: (45.92, 63.34)
  Dallas Fort Worth DFW: (5.24, -52.77)
  Denver DEN: (13.73, 80.23)


In [16]:
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": 35.32,
    "longitude": 75.38,
    "start_date":"2016-01-01",
    "end_date": "2018-12-31",
    "hourly": "temperature_2m",
}
timeout = 30

def get_airport_weather(start_date = '2016-01-01', end_date = '2018-12-31'):
    all_weather = []
    for airport_name, (lat, lon) in AIRPORTS.items():
        params= {
            'latitude': lat, 'longitude': lon, 'start_date': start_date, 'end_date': end_date,
            'daily': ['temperature_2m_max', 'temperature_2m_min', 'precipitation_sum', 'snowfall_sum',
                'windspeed_10m_max', 'weathercode', ],
            'time_zone': 'auto'}
        response = requests.get(url, params=params, timeout=timeout)
        data = response.json()
        daily = data["daily"]

        airport_df = pd.DataFrame({
        'airport_name': airport_name,
        'date': daily['time'],
        'temp_max_c': daily['temperature_2m_max'],
        'temp_min_c': daily['temperature_2m_min'],
        'precipitation_mm': daily['precipitation_sum'],
        'wind_speed_mph': daily['windspeed_10m_max'],
        'snow_mm': daily['snowfall_sum'],
        'weather_code': daily['weathercode'],
        
    })
        all_weather.append(airport_df)
    
    return pd.concat(all_weather, ignore_index=True)

In [17]:
airport_data  = get_airport_weather()
airport_weather = pd.DataFrame(airport_data)

In [20]:
takeoffs2016[takeoffs2016["DEP_TIME"].isna()][["FL_DATE", "DEP_DELAY", "DEP_TIME"]]


,FL_DATE,DEP_DELAY,DEP_TIME
699,2016-01-01,NaN,NaN
710,2016-01-01,NaN,NaN
713,2016-01-01,NaN,NaN
1231,2016-01-01,NaN,NaN
1242,2016-01-01,NaN,NaN
...,...,...,...
5611879,2016-12-31,NaN,NaN
5614160,2016-12-31,NaN,NaN
5614414,2016-12-31,NaN,NaN
5616755,2016-12-31,NaN,NaN
